In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA not available")

2.5.1+cu121
True
NVIDIA GeForce GTX 1060 6GB


In [3]:
import os
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold # Useremo i fold predefiniti, ma KFold è utile per capire la logica
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns # Per confusion matrix più belle
from tqdm.notebook import tqdm # Per progress bar nei loop

# --- Configurazioni Iniziali ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
# Percorsi (assicurati che siano corretti per il tuo ambiente)
BASE_DATA_PATH = "../data" # Modifica se necessario
META_FILE_PATH = os.path.join(BASE_DATA_PATH, "raw/UrbanSound8K.csv")
FEATURES_PATH = os.path.join(BASE_DATA_PATH, "processed/features") # Cartella contenente fold1, fold2, ... con i .npz

# Parametri del modello e training
NUM_CLASSES = 10  # UrbanSound8K ha 10 classi
LEARNING_RATE = 0.001
BATCH_SIZE = 32 # Puoi aggiustarlo in base alla memoria GPU
NUM_EPOCHS = 10 # Numero di epoche per ogni fold (da aggiustare)
FEATURE_TYPE = 'log_mel_spec' # 'log_mel_spec' o 'mfcc'
                               # Questo ti permetterà di testare con diverse feature come richiesto

# Per riproducibilità (opzionale ma consigliato)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Carica i metadati una volta
try:
    metadata_df = pd.read_csv(META_FILE_PATH)
except FileNotFoundError:
    print(f"Errore: File metadati non trovato in {META_FILE_PATH}. Verifica il percorso.")
    # In un notebook, potresti voler fermare l'esecuzione qui o gestire l'errore diversamente
    exit()

print(f"Metadati caricati. Numero totale di campioni: {len(metadata_df)}")

Metadati caricati. Numero totale di campioni: 8732


In [5]:
# Blocco 2: Definizione della Classe Dataset
# =========================================
# Questa classe carica le feature .npz e le etichette per il training/valutazione

class UrbanSoundFeaturesDataset(Dataset):
    def __init__(self, metadata_df, features_base_path, folds_to_include, feature_type='log_mel_spec'):
        """
        Args:
            metadata_df (pd.DataFrame): DataFrame con i metadati (slice_file_name, fold, classID).
            features_base_path (str): Percorso base alla cartella 'features' che contiene le sottocartelle 'foldX'.
            folds_to_include (list): Lista di numeri di fold da includere in questo dataset (es. [1,2,3] per training).
            feature_type (str): 'log_mel_spec' o 'mfcc' per selezionare quale feature caricare.
        """
        self.feature_type = feature_type
        self.file_paths = []
        self.labels = []

        # Determina la forma attesa delle feature (dovrai verificarla dai tuoi file .npz)
        # Ipotizziamo le dimensioni basate sulla tua precedente estrazione
        # Se diverse, il modello potrebbe aver bisogno di aggiustamenti o dovrai normalizzare qui le dimensioni
        # Esempio: (canali, n_mels/n_mfcc, n_frames)
        # self.expected_n_frames = 176 # Esempio, verifica questo valore!
        
        # Filtra i metadati per i fold specificati
        df_subset = metadata_df[metadata_df['fold'].isin(folds_to_include)]

        for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc=f"Caricamento dati per fold {folds_to_include}"):
            filename = row['slice_file_name']
            fold_num = row['fold']
            class_id = row['classID']
            
            # Costruisci il percorso al file .npz
            # Il nome del file .npz è lo stesso del file audio, ma senza estensione .wav e con .npz
            base_name = os.path.splitext(filename)[0]
            npz_path = os.path.join(features_base_path, f"fold{fold_num}", f"{base_name}.npz")

            if os.path.exists(npz_path):
                self.file_paths.append(npz_path)
                self.labels.append(class_id)
            else:
                print(f"Attenzione: file feature non trovato: {npz_path}")

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        npz_path = self.file_paths[idx]
        label = self.labels[idx]

        try:
            # Carica il file .npz
            data = np.load(npz_path)
            
            if self.feature_type == 'log_mel_spec':
                features = data['log_mel_spec'] # Shape attesa: (n_mels, n_frames)
            elif self.feature_type == 'mfcc':
                features = data['mfcc'] # Shape attesa: (n_mfcc, n_frames)
            else:
                raise ValueError(f"Tipo di feature non supportato: {self.feature_type}")

            # Esempio per verificare e aggiustare n_frames se necessario (non implementato qui per semplicità)
            # current_n_frames = features.shape[1]
            # if current_n_frames < self.expected_n_frames:
            #    pad_width = self.expected_n_frames - current_n_frames
            #    features = np.pad(features, ((0,0), (0, pad_width)), mode='constant')
            # elif current_n_frames > self.expected_n_frames:
            #    features = features[:, :self.expected_n_frames]


            # Il modello CNN si aspetta un input con una dimensione "canale"
            # Es. (batch_size, channels, height, width)
            # Quindi, se features è (H, W), lo trasformiamo in (1, H, W)
            if features.ndim == 2:
                features = np.expand_dims(features, axis=0)
            
            return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.long)
        
        except Exception as e:
            print(f"Errore durante il caricamento/processamento di {npz_path}: {e}")
            # Restituisci un campione dummy o gestisci l'errore come preferisci
            # Per semplicità, qui potremmo sollevare l'eccezione o restituire None
            # Se restituisci None, il DataLoader deve avere un collate_fn che gestisca i None
            # È meglio assicurarsi che tutti i file .npz siano validi
            return None, None # Il collate_fn dovrà gestire questo

# Funzione collate_fn personalizzata per gestire eventuali None da __getitem__
def collate_fn_skip_corrupted(batch):
    batch = list(filter(lambda x: x[0] is not None, batch))
    if not batch: # Se tutti i campioni nel batch sono corrotti
        return torch.empty(0), torch.empty(0) 
    return torch.utils.data.dataloader.default_collate(batch)

In [6]:
# Blocco 3: Definizione del Modello (CNN + GRU)
# ============================================

class CNNGRUClassifier(nn.Module):
    def __init__(self, num_classes, feature_type='log_mel_spec', cnn_out_channels=64, gru_hidden_size=128, gru_num_layers=1):
        super(CNNGRUClassifier, self).__init__()
        self.feature_type = feature_type

        # Determina l'altezza dell'input per la CNN (n_mels o n_mfcc)
        if feature_type == 'log_mel_spec':
            input_height = 128 # n_mels
        elif feature_type == 'mfcc':
            input_height = 20  # n_mfcc 
        else:
            raise ValueError("Tipo di feature non supportato")

        # --- Strati CNN ---
        # L'obiettivo è ridurre la dimensione della frequenza (altezza) e mantenere/trasformare quella temporale (larghezza)
        self.cnn_layers = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=(3, 3), padding=(1, 1)),
            nn.ReLU(),
            nn.BatchNorm2d(16),
            nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)), # Riduce altezza e larghezza

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3, 3), padding=(1, 1)),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)),

            nn.Conv2d(in_channels=32, out_channels=cnn_out_channels, kernel_size=(3, 3), padding=(1, 1)),
            nn.ReLU(),
            nn.BatchNorm2d(cnn_out_channels),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2,1)) # Riduce altezza, mantiene larghezza (tempo)
                                                         # Potresti voler usare (1,2) per ridurre il tempo prima della GRU
                                                         # o (2,2) e poi adattare. (2,1) è comune per comprimere le freq.
        )
        
        # Calcola la dimensione dell'output della CNN per la GRU
        # Questo è un calcolo approssimativo, sarebbe meglio fare un forward pass con un input dummy
        # Esempio di input dummy per calcolare le dimensioni:
        #   con dummy_input = torch.randn(1, 1, input_height, N_FRAMES)
        #   cnn_output_dummy = self.cnn_layers(dummy_input)
        #   self.cnn_output_freq_dim = cnn_output_dummy.shape[2]
        #   self.cnn_output_features = cnn_output_dummy.shape[1] * self.cnn_output_freq_dim
        # Per ora, lo facciamo in modo più dinamico nel forward o assumiamo una trasformazione
        
        # --- Strato GRU ---
        # L'input della GRU sarà (batch, seq_len, features)
        # seq_len è la dimensione temporale dopo la CNN
        # features è cnn_out_channels * altezza_residua_dopo_cnn
        # Per semplicità, usiamo AdaptiveMaxPool2d per ridurre la dimensione della frequenza a 1
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, None)) # Output: (batch, cnn_out_channels, 1, time_frames)
        
        self.gru = nn.GRU(
            input_size=cnn_out_channels, # Poiché comprimiamo la dimensione freq a 1
            hidden_size=gru_hidden_size,
            num_layers=gru_num_layers,
            batch_first=True, # Input: (batch, seq, feature)
            bidirectional=True
        )

        # --- Strato Fully Connected per Classificazione ---
        self.fc = nn.Linear(gru_hidden_size * 2, num_classes) # *2 per bidirezionale

    def forward(self, x):
        # x shape: (batch, 1, H_input, W_time)
        
        # Passaggio attraverso la CNN
        x = self.cnn_layers(x) # Shape: (batch, cnn_out_channels, H_cnn_out, W_cnn_out)
        
        # Prepara l'output della CNN per la GRU
        x = self.adaptive_pool(x) # Shape: (batch, cnn_out_channels, 1, W_cnn_out)
        x = x.squeeze(2)          # Shape: (batch, cnn_out_channels, W_cnn_out)
        x = x.permute(0, 2, 1)    # Shape: (batch, W_cnn_out, cnn_out_channels) -> (batch, seq_len, features)
        
        # Passaggio attraverso la GRU
        # self.gru.flatten_parameters() # Utile se si usa DataParallel o si hanno warning
        out, _ = self.gru(x) # out shape: (batch, seq_len, gru_hidden_size * num_directions)
        
        # Prendi l'output dell'ultimo time step della GRU (o fai pooling/attention)
        # Qui usiamo l'ultimo output
        x = out[:, -1, :] # Shape: (batch, gru_hidden_size * num_directions)
        
        # Passaggio attraverso lo strato fully connected
        x = self.fc(x) # Shape: (batch, num_classes)
        
        return x

# Test rapido della forma del modello (opzionale, ma utile per debug)
# Questo presuppone che tu sappia N_FRAMES
# N_FRAMES_EXAMPLE = 176 # Sostituisci con il numero di frame effettivo dai tuoi .npz
# dummy_model_test = CNNGRUClassifier(num_classes=NUM_CLASSES, feature_type=FEATURE_TYPE)
# dummy_input_test = torch.randn(BATCH_SIZE, 1, 128 if FEATURE_TYPE=='log_mel_spec' else 20, N_FRAMES_EXAMPLE) 
# output_test = dummy_model_test(dummy_input_test)
# print(f"Forma output modello: {output_test.shape}") # Atteso: (BATCH_SIZE, NUM_CLASSES)

In [7]:
# Blocco 4: Funzione di Training e Valutazione per Epoca
# =====================================================

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train() # Imposta il modello in modalità training
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for inputs, labels in tqdm(dataloader, desc="Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # Azzera i gradienti

        outputs = model(inputs) # Forward pass
        loss = criterion(outputs, labels) # Calcola la loss
        
        loss.backward() # Backward pass
        optimizer.step() # Aggiorna i pesi

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct_predictions += torch.sum(preds == labels.data)
        total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc = correct_predictions.double() / total_samples
    return epoch_loss, epoch_acc.item()


def evaluate_model(model, dataloader, criterion, device):
    model.eval() # Imposta il modello in modalità valutazione
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    all_preds = []
    all_labels = []

    with torch.no_grad(): # Disabilita il calcolo dei gradienti durante la valutazione
        for inputs, labels in tqdm(dataloader, desc="Evaluating", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_predictions += torch.sum(preds == labels.data)
            total_samples += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total_samples
    epoch_acc = correct_predictions.double() / total_samples
    return epoch_loss, epoch_acc.item(), all_labels, all_preds

In [8]:
# Blocco 5: Funzioni Ausiliarie (es. per metriche dettagliate)
# ============================================================

# Se vuoi usare Focal Loss, la sua implementazione andrebbe qui.
# Per ora usiamo CrossEntropyLoss.

def plot_confusion_matrix(cm, class_names, fold_num):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.title(f'Confusion Matrix - Fold {fold_num}')
    plt.show()

# Ottieni i nomi delle classi dai metadati per le visualizzazioni
class_names = sorted(metadata_df['class'].unique())

In [ ]:
# Blocco 6: Loop di K-Fold Cross-Validation
# =========================================
# UrbanSound8K è già diviso in 10 fold. Useremo questa suddivisione.

num_folds = 10 # Come da dataset UrbanSound8K
fold_results = [] # Per salvare i risultati di ogni fold

# Assicurati che FEATURES_PATH sia corretto e contenga le cartelle fold1, fold2, ...
if not os.path.exists(FEATURES_PATH):
    print(f"ERRORE: La cartella delle feature '{FEATURES_PATH}' non esiste.")
    print("Assicurati di aver eseguito lo script di preprocessing e che il percorso sia corretto.")
    # Interrompi se le feature non ci sono
    # exit() # In un notebook, puoi semplicemente non eseguire le celle successive

for fold_idx in range(1, num_folds + 1):
    print(f"\n--- Inizio Fold {fold_idx}/{num_folds} ---")

    # Definisci i fold per training e test
    # Per UrbanSound8K, di solito si usa un fold per test e gli altri per training.
    # Non stiamo usando un validation set separato qui per semplicità,
    # ma potresti volerlo fare (es. fold_idx-1 per validation, se fold_idx > 1).
    train_fold_numbers = [f for f in range(1, num_folds + 1) if f != fold_idx]
    test_fold_numbers = [fold_idx]

    print(f"Fold di Training: {train_fold_numbers}")
    print(f"Fold di Test: {test_fold_numbers}")

    # --- Creazione Dataset e DataLoader ---
    train_dataset = UrbanSoundFeaturesDataset(metadata_df, FEATURES_PATH, train_fold_numbers, FEATURE_TYPE)
    test_dataset = UrbanSoundFeaturesDataset(metadata_df, FEATURES_PATH, test_fold_numbers, FEATURE_TYPE)

    # Verifica che i dataset non siano vuoti
    if len(train_dataset) == 0:
        print(f"Errore: Train dataset per il fold {fold_idx} è vuoto. Controlla i percorsi e i file .npz.")
        continue 
    if len(test_dataset) == 0:
        print(f"Errore: Test dataset per il fold {fold_idx} è vuoto.")
        continue

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_fn_skip_corrupted)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn_skip_corrupted)
    
    # --- Inizializzazione Modello, Loss, Optimizer ---
    model = CNNGRUClassifier(num_classes=NUM_CLASSES, feature_type=FEATURE_TYPE).to(DEVICE)
    criterion = nn.CrossEntropyLoss() # Puoi sostituirla con FocalLoss
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # Scheduler (opzionale, per ridurre il learning rate durante il training)
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5, verbose=True)

    # --- Ciclo di Training per il Fold Corrente ---
    best_test_acc = 0.0 # Se vuoi salvare il miglior modello per fold
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        # Per la valutazione sul test set alla fine di ogni epoca (può portare a overfitting sulla policy di stop)
        # È meglio avere un validation set separato per questo e per lo scheduler
        # Ma per semplicità, valutiamo sul test set del fold qui
        test_loss, test_acc, _, _ = evaluate_model(model, test_loader, criterion, DEVICE) # Ignoriamo labels e preds qui

        print(f"Fold {fold_idx} Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}")
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        
        # if scheduler:
        #    scheduler.step(test_loss) # O validation_loss se hai un validation set

        # Salva il modello se migliora l'accuratezza sul test set (o validation)
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            # torch.save(model.state_dict(), f"best_model_fold_{fold_idx}_{FEATURE_TYPE}.pth")
            # print(f"Miglior modello per fold {fold_idx} salvato con acc: {best_test_acc:.4f}")


    # --- Valutazione Finale sul Test Set per questo Fold ---
    print(f"\nValutazione finale per Fold {fold_idx} sul Test Set:")
    final_test_loss, final_test_acc, all_true_labels, all_predicted_labels = evaluate_model(model, test_loader, criterion, DEVICE)
    
    f1_macro = f1_score(all_true_labels, all_predicted_labels, average='macro', zero_division=0)
    f1_weighted = f1_score(all_true_labels, all_predicted_labels, average='weighted', zero_division=0)
    
    print(f"Fold {fold_idx} - Test Loss: {final_test_loss:.4f}, Test Accuracy: {final_test_acc:.4f}")
    print(f"Fold {fold_idx} - F1 Score (Macro): {f1_macro:.4f}, F1 Score (Weighted): {f1_weighted:.4f}")
    
    # Classification Report
    report = classification_report(all_true_labels, all_predicted_labels, target_names=class_names, zero_division=0)
    print(f"\nClassification Report per Fold {fold_idx}:\n{report}")
    
    # Confusion Matrix
    cm = confusion_matrix(all_true_labels, all_predicted_labels, labels=range(NUM_CLASSES))
    plot_confusion_matrix(cm, class_names, fold_idx)
    
    fold_results.append({
        'fold': fold_idx,
        'accuracy': final_test_acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'loss': final_test_loss,
        'report': report,
        'cm': cm,
        'history': history # Salva la storia per plottare le curve di apprendimento
    })
    
    # Libera memoria GPU se necessario (opzionale)
    del model, train_loader, test_loader, train_dataset, test_dataset
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


--- Inizio Fold 1/10 ---
Fold di Training: [2, 3, 4, 5, 6, 7, 8, 9, 10]
Fold di Test: [1]


Caricamento dati per fold [2, 3, 4, 5, 6, 7, 8, 9, 10]:   0%|          | 0/7859 [00:00<?, ?it/s]

Caricamento dati per fold [1]:   0%|          | 0/873 [00:00<?, ?it/s]

Training:   0%|          | 0/246 [00:00<?, ?it/s]

In [ ]:
# Blocco 7: Visualizzazione dei Risultati Complessivi
# ===================================================

if fold_results:
    print("\n\n--- Risultati Complessivi della Cross-Validation ---")
    
    accuracies = [res['accuracy'] for res in fold_results]
    f1_macros = [res['f1_macro'] for res in fold_results]
    f1_weighteds = [res['f1_weighted'] for res in fold_results]

    mean_accuracy = np.mean(accuracies)
    std_accuracy = np.std(accuracies)
    mean_f1_macro = np.mean(f1_macros)
    std_f1_macro = np.std(f1_macros)
    mean_f1_weighted = np.mean(f1_weighteds)
    std_f1_weighted = np.std(f1_weighteds)

    print(f"\nFeature Type Utilizzato: {FEATURE_TYPE}")
    print(f"Accuratezza Media: {mean_accuracy:.4f} +/- {std_accuracy:.4f}")
    print(f"F1 Score Macro Medio: {mean_f1_macro:.4f} +/- {std_f1_macro:.4f}")
    print(f"F1 Score Weighted Medio: {mean_f1_weighted:.4f} +/- {std_f1_weighted:.4f}")

    print("\nAccuratezze per Fold:")
    for i, acc in enumerate(accuracies):
        print(f"Fold {i+1}: {acc:.4f}")

    # Puoi anche plottare le curve di apprendimento medie o per ogni fold
    # Esempio: plotta la curva di test accuracy media
    plt.figure(figsize=(12, 6))
    for i, res in enumerate(fold_results):
        plt.plot(res['history']['test_acc'], label=f'Fold {i+1} Test Acc', alpha=0.5)
    
    # Calcola la media delle curve di test_acc
    if NUM_EPOCHS > 0:
        avg_test_acc_curve = np.mean([res['history']['test_acc'] for res in fold_results if len(res['history']['test_acc']) == NUM_EPOCHS], axis=0)
        if avg_test_acc_curve.ndim > 0 and len(avg_test_acc_curve) > 0 : # Assicurati che ci sia qualcosa da plottare
             plt.plot(avg_test_acc_curve, label='Avg Test Acc', linewidth=3, color='black')
    
    plt.title(f'Curve di Test Accuracy per Fold ({FEATURE_TYPE})')
    plt.xlabel('Epoca')
    plt.ylabel('Accuratezza')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()

else:
    print("Nessun risultato dai fold da visualizzare. Controlla eventuali errori precedenti.")